# Multi-Task CSAT + CES Prediction
## AI-Driven Customer Experience Modeling

**Dataset**: Amazon Consumer Reviews (`7817_1.csv`)  
**Model**: BERT-base shared encoder with two independent classification heads  
**Tasks**:
- **CSAT** (Customer Satisfaction) — derived from `reviews.rating` and `reviews.doRecommend`  
  → 3 classes: `Dissatisfied` | `Neutral` | `Satisfied`
- **CES** (Customer Effort Score) — derived from effort/friction keyword signals in review text  
  → 2 classes: `Easy` | `Difficult`

---

## 0. Setup & Imports

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')

# Add project directory to path so we can import modules
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {"CUDA" if torch.cuda.is_available() else "CPU"}')

## 1. Exploratory Data Analysis

In [ ]:
CSV_PATH = '/Users/vedh/Desktop/csat proj  2/7817_1.csv'

df_raw = pd.read_csv(CSV_PATH, low_memory=False, on_bad_lines='skip')
print(f'Total rows : {len(df_raw):,}')
print(f'Columns    : {list(df_raw.columns)}')
df_raw[['reviews.text', 'reviews.title', 'reviews.rating', 'reviews.doRecommend']].head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rating distribution
df_raw['reviews.rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution (1–5)', fontweight='bold')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Count')

# doRecommend distribution
df_raw['reviews.doRecommend'].value_counts().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Do Recommend?', fontweight='bold')
axes[1].set_xlabel('Value'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Review text length distribution
lengths = df_raw['reviews.text'].dropna().str.split().apply(len)
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(lengths, bins=60, color='mediumseagreen', edgecolor='white')
ax.set_title('Review Length Distribution (words)', fontweight='bold')
ax.set_xlabel('Word count'); ax.set_ylabel('Reviews')
ax.axvline(lengths.median(), color='red', linestyle='--', label=f'Median={lengths.median():.0f}')
ax.legend()
plt.tight_layout(); plt.show()
print(f'Median review length: {lengths.median():.0f} words | Max: {lengths.max()} | 95th pct: {lengths.quantile(0.95):.0f}')

## 2. Label Engineering

In [ ]:
from data_prep import load_dataset, CSAT_LABEL_MAP, CES_LABEL_MAP

df = load_dataset(CSV_PATH)
print(df.head())
print(f'\nDataset shape: {df.shape}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

csat_counts = df['csat_label'].map(CSAT_LABEL_MAP).value_counts()
csat_counts.plot(kind='bar', ax=axes[0], color=['#e74c3c','#f39c12','#27ae60'], edgecolor='white')
axes[0].set_title('CSAT Label Distribution', fontweight='bold')
axes[0].set_xlabel(''); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

ces_counts = df['ces_label'].map(CES_LABEL_MAP).value_counts()
ces_counts.plot(kind='bar', ax=axes[1], color=['#3498db','#e74c3c'], edgecolor='white')
axes[1].set_title('CES Label Distribution', fontweight='bold')
axes[1].set_xlabel(''); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()

In [ ]:
# CSAT × CES co-occurrence heatmap
cross = pd.crosstab(
    df['csat_label'].map(CSAT_LABEL_MAP),
    df['ces_label'].map(CES_LABEL_MAP)
)
plt.figure(figsize=(6,4))
sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd')
plt.title('CSAT × CES Co-occurrence', fontweight='bold')
plt.tight_layout(); plt.show()
print('Note: High CSAT + Difficult CES = reviews with both positive sentiment and friction signals.')

## 3. Build DataLoaders

In [ ]:
from data_prep import build_dataloaders

# For notebook demo, use max_rows=3000 for speed
# Set max_rows=None to use the full dataset
train_loader, val_loader, test_loader, tokenizer = build_dataloaders(
    csv_path   = CSV_PATH,
    model_name = 'bert-base-uncased',
    max_len    = 256,
    batch_size = 16,
    max_rows   = 3000         # ← change to None for full run
)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

# Peek at a single batch
sample_batch = next(iter(train_loader))
print('\nSample batch:')
for k, v in sample_batch.items():
    print(f'  {k}: {v.shape} | dtype={v.dtype}')

## 4. Model Architecture

In [ ]:
from model import MultiTaskReviewModel, CSAT_CLASSES, CES_CLASSES

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = MultiTaskReviewModel(model_name='bert-base-uncased', dropout=0.3).to(device)

# Count parameters
total_params    = sum(p.numel() for p in model.parameters())
trainable_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'CSAT classes : {CSAT_CLASSES}')
print(f'CES  classes : {CES_CLASSES}')

In [ ]:
# Quick forward pass check
ids  = sample_batch['input_ids'].to(device)
mask = sample_batch['attention_mask'].to(device)

with torch.no_grad():
    csat_logits, ces_logits = model(ids, mask)

print(f'CSAT logits shape: {csat_logits.shape}')  # (16, 3)
print(f'CES  logits shape: {ces_logits.shape}')   # (16, 2)
print('Forward pass OK ✓')

## 5. Training

> **Note**: Training BERT fine-tuning for 5 epochs over the full dataset takes ~1-2 hours on CPU.  
> Use a GPU runtime for faster training, or reduce `max_rows` for a quick demo run.

To train from the command line:  
```bash
cd multitask_model
python train.py
```

In [ ]:
# ─── Quick demo training (1 epoch, 500 rows) ──────────────────────────────
# Remove or modify to run the full training

import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

# Small demo loaders
demo_train, demo_val, demo_test, _ = build_dataloaders(
    csv_path=CSV_PATH, max_len=128, batch_size=8, max_rows=500
)

demo_model = MultiTaskReviewModel().to(device)
optimizer  = AdamW(demo_model.parameters(), lr=2e-5)
loss_fn    = nn.CrossEntropyLoss()

demo_model.train()
epoch_loss = 0
for step, batch in enumerate(demo_train):
    ids  = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    cl   = batch['csat_label'].to(device)
    el   = batch['ces_label'].to(device)

    optimizer.zero_grad()
    csat_logits, ces_logits = demo_model(ids, mask)
    loss = loss_fn(csat_logits, cl) + loss_fn(ces_logits, el)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()
    if (step+1) % 5 == 0:
        print(f'Step {step+1}/{len(demo_train)} | Loss: {epoch_loss/(step+1):.4f}')

print('\nDemo training complete ✓')

## 6. Load Best Checkpoint & Evaluate

Run this cell after running `python train.py` to completion.

In [ ]:
from pathlib import Path
import torch.nn.functional as F

CKPT = Path('./checkpoints/best_model.pt')

if CKPT.exists():
    ckpt = torch.load(CKPT, map_location=device)
    trained_model = MultiTaskReviewModel().to(device)
    trained_model.load_state_dict(ckpt['model_state'])
    trained_model.eval()
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
    print('Val metrics at best checkpoint:')
    for k, v in ckpt['val_metrics'].items():
        print(f'  {k}: {v:.4f}')
else:
    print('No checkpoint found. Run: python train.py first.')

## 7. Inference on Sample Reviews

In [ ]:
# ── Requires trained checkpoint ──────────────────────────────────────────
if not CKPT.exists():
    print('Train the model first. Running with untrained model for shape demonstration.')
    trained_model = MultiTaskReviewModel().to(device)

sample_reviews = [
    'The product is absolutely fantastic. Fast delivery and great packaging.',
    'The product is great but the return process was incredibly frustrating.',
    'Terrible quality. Broke after two days. Customer service was useless.',
    'Decent product, took a while to arrive but works as expected.',
    'I love this! Super easy to use. Exactly what I needed.',
    'The item was damaged and I had to wait 3 weeks for a replacement.',
]

trained_model.eval()
results = []
for review in sample_reviews:
    enc = tokenizer(review, truncation=True, max_length=256,
                    padding='max_length', return_tensors='pt')
    with torch.no_grad():
        clogits, elogits = trained_model(
            enc['input_ids'].to(device),
            enc['attention_mask'].to(device)
        )
    csat = CSAT_CLASSES[clogits.argmax(-1).item()]
    ces  = CES_CLASSES[elogits.argmax(-1).item()]
    results.append({'Review (truncated)': review[:60]+'…', 'CSAT': csat, 'CES': ces})

pd.DataFrame(results)

## 8. Training Curves & Visualisation

In [ ]:
hist_path = Path('./checkpoints/training_history.json')

if hist_path.exists():
    with open(hist_path) as f:
        data = json.load(f)
    hist = data['history']

    ep   = [h['epoch']       for h in hist]
    tl   = [h['train_loss']  for h in hist]
    vl   = [h['val_loss']    for h in hist]
    csat_f1 = [h['val_csat_f1'] for h in hist]
    ces_f1  = [h['val_ces_f1']  for h in hist]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle('Multi-Task Training Curves', fontsize=13, fontweight='bold')

    axes[0].plot(ep, tl, 'o-', label='Train Loss')
    axes[0].plot(ep, vl, 's-', label='Val Loss')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(ep, csat_f1, 'o-', color='#e74c3c', label='CSAT F1')
    axes[1].plot(ep, ces_f1,  's-', color='#3498db', label='CES  F1')
    axes[1].set_title('Validation Weighted F1'); axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout(); plt.show()

    print('\nFinal test metrics:')
    for k, v in data['test'].items():
        print(f'  {k}: {v:.4f}')
else:
    print('No training history found. Run: python train.py')